### Chunking

In [10]:
from typing import List

def split_into_chunks(doc_file: str) -> List[str]:
    with open(doc_file, 'r') as file:
        content = file.read()

    return [chunk for chunk in content.split("\n\n")]

chunks = split_into_chunks("doc.md")

for i, chunk in enumerate(chunks):
    print(f"[{i}] {chunk}\n")

[0] # Project Aether: Full Technical & Operational Manual (v4.2.1)

[1] ## 1. Mission Objectives and Strategic Overview
Project Aether is a sub-orbital drone initiative designed to solve "last-mile" medical logistics in extreme geographic environments. The primary objective is the rapid transport of temperature-sensitive vaccines, blood bags, and emergency surgical kits to regions where traditional ground or rotor-wing transport is impossible due to terrain or weather.

[2] ## 2. Propulsion and the Vortex-7 Engine
The heart of the Aether UAV is the Vortex-7 propulsion system. Unlike standard electric drones, the Vortex-7 is a hybrid-staged combustion engine that utilizes a unique propellant mix of liquid hydrogen and compressed oxygen. This allows the drone to operate in the "Thin-Air" zone where standard propellers lose over 60% of their aerodynamic efficiency.

[3] ## 3. Flight Performance and Atmospheric Metrics
Project Aether is engineered for high-altitude endurance. The drone mai

### Embedding

In [12]:
from sentence_transformers import SentenceTransformer

# embedding_model = SentenceTransformer("shibing624/text2vec-base-chinese")
embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

def embed_chunk(chunk: str) -> List[float]:
    embedding = embedding_model.encode(chunk, normalize_embeddings=True)
    return embedding.tolist()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8230.94it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
embeddings = [embed_chunk(chunk) for chunk in chunks]

print(len(embeddings))
print(embeddings[0])

26
[0.00602783914655447, 0.0063348389230668545, -0.009432416409254074, 0.01402759924530983, -0.03374600037932396, -0.002329421229660511, -0.00787561759352684, -0.012955617159605026, 0.010687494650483131, 0.030590845271945, -0.0033694126177579165, -0.0034477601293474436, 0.02441341057419777, -0.0488223172724247, -0.0244494266808033, -0.002697504824027419, -0.04537718743085861, -0.029339991509914398, -0.07893693447113037, 0.014458860270678997, 0.0101703442633152, 0.0036650372203439474, -0.04837070032954216, -0.026795653626322746, -0.027626553550362587, 0.02366609498858452, 0.03515605255961418, -0.016378166154026985, 0.10663190484046936, 0.06170070171356201, 0.006334294099360704, -0.0046724118292331696, 0.03785862401127815, -0.03415212407708168, -0.019089892506599426, -0.049256157130002975, 0.024484140798449516, 0.0024335391353815794, -0.00883020181208849, -0.06649857759475708, 0.03496372327208519, -0.02229837328195572, -0.0020144658628851175, -0.05209960788488388, -0.061741337180137634, 

In [15]:
import chromadb

chromadb_client = chromadb.EphemeralClient()

# 1. Delete the old collection if it exists to reset the dimensions
try:
    chromadb_client.delete_collection(name="default")
except:
    pass 

# 2. This will now create a fresh collection with 1024-dimension support
chromadb_collection = chromadb_client.get_or_create_collection(name="default")

# 3. Optimized batch-save function (much faster than a loop)
def save_embeddings(chunks: List[str], embeddings: List[List[float]]) -> None:
    chromadb_collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(chunks))]
    )

save_embeddings(chunks, embeddings)

### Retrieve

In [16]:
def retrieve(query: str, top_k: int) -> List[str]:
    query_embedding = embed_chunk(query)
    results = chromadb_collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]

query = "What is the required part number for the intake filters used in the Atacama Desert testing corridor, and how often must they be cleaned?"
retrieved_chunks = retrieve(query, 5)

for i, chunk in enumerate(retrieved_chunks):
    print(f"[{i}] {chunk}\n")

[0] ## 10. Global Deployment: Atacama Desert Stress Testing
The Atacama Desert in Chile serves as the primary site for high-altitude stress testing. Due to the extreme dryness and fine particulate dust, Aether units in this region are equipped with "Sand-Shield" intake filters (Part #SS-88). These filters must be cleaned manually every 48 hours to prevent intake choking.

[1] ## 8. Maintenance and Lubrication Requirements
To maintain the integrity of the Vortex-7 engine, a specific "Class-C" inspection must be performed every 250 flight hours. A key technical requirement is the use of "Aether-Lube 400" synthetic lubricant for the central turbine bearings. Standard petroleum-based lubricants are strictly prohibited as they tend to solidify at -55°C.

[2] ## 20. Component Lifespan: Central Turbine
The central turbine of the Vortex-7 engine has a total service life of 2,500 flight hours. At the 2,000-hour mark, the turbine blades must undergo a "Stress-Fracture Acoustic Test" (SFAT) to ch

### Rerank

In [17]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2')

def rerank(query: str, retrieved_chunks: List[str], top_k: int) -> List[str]:
    # cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    pairs = [(query, chunk) for chunk in retrieved_chunks]
    scores = cross_encoder.predict(pairs)

    scored_chunks = list(zip(retrieved_chunks, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return [chunk for chunk, _ in scored_chunks][:top_k]

reranked_chunks = rerank(query, retrieved_chunks, 3)

for i, chunk in enumerate(reranked_chunks):
    print(f"[{i}] {chunk}\n")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 23942.95it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0] ## 10. Global Deployment: Atacama Desert Stress Testing
The Atacama Desert in Chile serves as the primary site for high-altitude stress testing. Due to the extreme dryness and fine particulate dust, Aether units in this region are equipped with "Sand-Shield" intake filters (Part #SS-88). These filters must be cleaned manually every 48 hours to prevent intake choking.

[1] ## 8. Maintenance and Lubrication Requirements
To maintain the integrity of the Vortex-7 engine, a specific "Class-C" inspection must be performed every 250 flight hours. A key technical requirement is the use of "Aether-Lube 400" synthetic lubricant for the central turbine bearings. Standard petroleum-based lubricants are strictly prohibited as they tend to solidify at -55°C.

[2] ## 20. Component Lifespan: Central Turbine
The central turbine of the Vortex-7 engine has a total service life of 2,500 flight hours. At the 2,000-hour mark, the turbine blades must undergo a "Stress-Fracture Acoustic Test" (SFAT) to ch

### Testing

In [ ]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
google_client = genai.Client()

def generate(query: str, chunks: List[str]) -> str:
    # Pre-process chunks into a numbered list to avoid f-string join errors
    context_text = "\n\n".join([f"[Fragment {i+1}]: {chunk}" for i, chunk in enumerate(chunks)])
    
    prompt = f"""You are a highly precise technical assistant. Your task is to answer the user's question based ONLY on the provided document fragments.

    Guidelines:
    1. Use the provided context to form a factual and concise answer.
    2. If the answer is not contained within the context, state that you do not know. Do not hallucinate or use outside knowledge.
    3. Mention specific technical terms, part numbers, or metrics if they are relevant to the answer.

    Context Fragments:
    {context_text}

    User Question: {query}

    Answer:"""

    print(f"--- QUESTION: {query} ---\n")

    response = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

# --- Testing Questions ---
test_questions = [
    "What is the required part number for the intake filters used in the Atacama Desert testing corridor, and how often must they be cleaned?",
    "What specific logic does the drone default to if its three Node-Core units disagree on a flight path in software version 4.2?",
    "If the drone's external status LED shows a '3-short, 1-long' red flash pattern, what is the diagnosed issue and what is the required fix?"
]

# Loop through questions to test your RAG system
for i, q in enumerate(test_questions):
    print(f"\nTEST CASE {i+1}:")
    # Step 1: Retrieve (from your previous cells)
    retrieved = retrieve(q, 5) 
    # Step 2: Rerank (from your previous cells)
    reranked = rerank(q, retrieved, 3)
    # Step 3: Generate
    result = generate(q, reranked)
    print(f"RESULT:\n{result}\n" + "="*50)


TEST CASE 1:
--- Question: What is the required part number for the intake filters used in the Atacama Desert testing corridor, and how often must they be cleaned? ---

RESULT:
The required part number for the intake filters used in the Atacama Desert is #SS-88. These "Sand-Shield" intake filters must be cleaned manually every 48 hours.

TEST CASE 2:
--- Question: What specific logic does the drone default to if its three Node-Core units disagree on a flight path in software version 4.2? ---

RESULT:
If the three Node-Core units disagree on a flight path adjustment, the system defaults to the "Last Known Safe Path" (LKSP) until human intervention occurs.

TEST CASE 3:
--- Question: If the drone's external status LED shows a '3-short, 1-long' red flash pattern, what is the diagnosed issue and what is the required fix? ---

RESULT:
If the drone's external status LED shows a '3-short, 1-long' red flash pattern, the diagnosed issue is a "Pressure-Sensor Mismatch" in the fuel lines. The re